# MVP Playground

This notebook is a self-contained first-pass prototype for the human / robot / AI music project.

It is intentionally MIDI-only and rule-based:

- load an input MIDI file
- extract a small set of musical features
- choose a response strategy
- generate a reactive MIDI response
- write the result to disk


In [1]:
from pathlib import Path
import numpy as np
import pretty_midi

SEED = 7
np.random.seed(SEED)

def find_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root from current working directory')

ROOT = find_repo_root()
INPUT_MIDI = ROOT / 'data' / 'input_midi' / 'mvp_minimalist_input.mid'
OUTPUT_MIDI = ROOT / 'data' / 'output_midi' / 'mvp_minimalist_response.mid'
TEMPO = 120
TIME_SIG = (4, 4)
LATENCY_BARS = 1

INPUT_MIDI, OUTPUT_MIDI


(PosixPath('/Users/adrientalbot/Desktop/ai-jam-partner/data/input_midi/mvp_minimalist_input.mid'),
 PosixPath('/Users/adrientalbot/Desktop/ai-jam-partner/data/output_midi/mvp_minimalist_response.mid'))

## Load Input

The MVP assumes a single MIDI track for the first pass, but it will fall back to the first track if multiple instruments are present.


In [2]:
midi = pretty_midi.PrettyMIDI(str(INPUT_MIDI))
if not midi.instruments:
    raise ValueError('Input MIDI contains no instruments')

instrument = midi.instruments[0]
notes = sorted(instrument.notes, key=lambda n: n.start)
source_instrument_name = pretty_midi.program_to_instrument_name(instrument.program)

print(f'Instruments: {len(midi.instruments)}')
print(f'Source instrument: {source_instrument_name}')
print(f'Notes: {len(notes)}')
print(f'Pitch range: {min(n.pitch for n in notes)}-{max(n.pitch for n in notes)}')
print(f'Duration: {midi.get_end_time():.2f}s')


Instruments: 1
Source instrument: Acoustic Grand Piano
Notes: 176
Pitch range: 36-84
Duration: 63.80s


## Feature Extraction

The first version only needs a small musical summary: density, register, phrase length, and a short motif from the tail of the phrase.


In [3]:
def seconds_per_bar(tempo=TEMPO, time_sig=TIME_SIG):
    return 60.0 / tempo * time_sig[0]

def density_bucket(note_count):
    if note_count < 24:
        return 'low'
    if note_count < 64:
        return 'medium'
    return 'high'

def register_bucket(avg_pitch):
    if avg_pitch < 50:
        return 'low'
    if avg_pitch < 70:
        return 'mid'
    return 'high'

def extract_features(notes, tempo=TEMPO, time_sig=TIME_SIG):
    pitches = [n.pitch for n in notes]
    avg_pitch = float(np.mean(pitches))
    duration = notes[-1].end - notes[0].start if notes else 0.0
    bars = max(1, int(round(duration / seconds_per_bar(tempo, time_sig))))
    tail = sorted(notes, key=lambda n: n.start)[-4:]
    motif = [n.pitch for n in tail]
    return {
        'note_count': len(notes),
        'avg_pitch': avg_pitch,
        'density': density_bucket(len(notes)),
        'register': register_bucket(avg_pitch),
        'bars': bars,
        'motif': motif,
    }

features = extract_features(notes)
features


{'note_count': 176,
 'avg_pitch': 63.97727272727273,
 'density': 'high',
 'register': 'mid',
 'bars': 32,
 'motif': [72, 76, 79, 84]}

In [4]:
MEMORY = {
    "notes": [],
    "last_motif": None,
    "max_bars": 4
}

def quantize(t, step):
    return round(t / step) * step

def update_memory(memory, new_notes, tempo, time_sig):
    bar_len = seconds_per_bar(tempo, time_sig)
    max_time = memory["max_bars"] * bar_len

    memory["notes"].extend(new_notes)

    if memory["notes"]:
        latest_time = max(n.end for n in memory["notes"])
        memory["notes"] = [
            n for n in memory["notes"]
            if n.end >= latest_time - max_time
        ]

from collections import Counter

def detect_key(notes):
    if not notes:
        return 60
    pcs = [n.pitch % 12 for n in notes]
    root = Counter(pcs).most_common(1)[0][0]
    return 60 + root

def get_major_scale(root):
    return [(root + i) % 12 for i in [0, 2, 4, 5, 7, 9, 11]]

def constrain_to_scale(pitches, scale_pc):
    out = []
    for p in pitches:
        pc = p % 12
        if pc not in scale_pc:
            closest = min(scale_pc, key=lambda x: abs(x - pc))
            p = p - pc + closest
        out.append(p)
    return out

## Response Policy

This is a simple rule table for the MVP. It is not a learned model.


In [5]:
def decide_action(features):
    density = features['density']
    register = features['register']

    if density == 'high':
        mode = 'contrast'
        response_density = 'low'
    elif density == 'medium' and register == 'high':
        mode = 'fragment'
        response_density = 'medium'
    elif density == 'medium':
        mode = 'sequence'
        response_density = 'medium'
    else:
        mode = 'repeat'
        response_density = 'low'

    return {
        'mode': mode,
        'response_density': response_density,
        'bars': features['bars'],
        'latency_bars': LATENCY_BARS,
        'target_notes': 32,
    }

action = decide_action(features)
action


{'mode': 'contrast',
 'response_density': 'low',
 'bars': 32,
 'latency_bars': 1,
 'target_notes': 32}

## Generate Response

The response starts after one bar of latency and uses a simple D minor palette for the first prototype.


In [6]:
def choose_response_program(source_instrument, action, features):
    # Keep the same instrument as the source for this MVP.
    return source_instrument.program


def build_response(notes, action, source_instrument, features, tempo=TEMPO, time_sig=TIME_SIG, start_time=0.0):
    response = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    response_program = choose_response_program(source_instrument, action, features)
    inst = pretty_midi.Instrument(program=response_program)

    # --- Combine memory + current notes ---
    all_notes = MEMORY["notes"] + notes
    recent_notes = sorted(all_notes, key=lambda n: n.start)[-8:]
    motif = [n.pitch for n in recent_notes[-4:]] if recent_notes else []

    # --- Harmonic context ---
    key_root = detect_key(all_notes)
    scale_pc = get_major_scale(key_root)

    # --- Timing ---
    bar_len = seconds_per_bar(tempo, time_sig)
    step = (60.0 / tempo) / 2  # 8th notes
    t = quantize(start_time + action['latency_bars'] * bar_len, step)

    note_spacing = step

    # --- Density-driven note length ---
    if action['response_density'] == 'low':
        note_length = step * 2
    elif action['response_density'] == 'high':
        note_length = step * 0.75
    else:
        note_length = step * 1.2

    # --- Motif logic ---
    if action['mode'] == 'repeat' and MEMORY.get("last_motif"):
        pitches = MEMORY["last_motif"]

    elif action['mode'] == 'fragment' and len(motif) >= 2:
        pitches = motif[-2:] * 2

    elif action['mode'] == 'sequence' and motif:
        pitches = motif + [p + 2 for p in motif]

    else:
        # harmonic fallback
        intervals = [0, 2, 4, 7]
        pitches = [
            key_root + np.random.choice(intervals)
            for _ in range(max(4, len(motif) or 4))
        ]

    # --- Contrast ---
    if action['mode'] == 'contrast':
        pitches = [p - 12 for p in pitches]

    # --- Register alignment ---
    if notes:
        avg_pitch = int(np.mean([n.pitch for n in notes]))
        pitches = [p + (avg_pitch - 60) for p in pitches]

    # --- Constrain to scale ---
    pitches = constrain_to_scale(pitches, scale_pc)

    # --- Density trimming ---
    if action['response_density'] == 'low':
        pitches = pitches[:max(4, len(pitches) // 2)]

    # --- Phrase expansion ---
    base_pattern = pitches[:] if pitches else [key_root]
    target_notes = action.get('target_notes', 32)
    cycle = 0
    while len(pitches) < target_notes:
        transposition = 0 if cycle % 2 == 0 else 2
        for p in base_pattern:
            if len(pitches) >= target_notes:
                break
            pitches.append(p + transposition)
        cycle += 1

    pitches = pitches[:target_notes]

    # --- Velocity shaping ---
    base_vel = 70
    velocities = [
        base_vel + int(10 * np.sin(i))
        for i in range(len(pitches))
    ]

    # --- Create notes ---
    for idx, pitch in enumerate(pitches):
        note = pretty_midi.Note(
            velocity=int(np.clip(velocities[idx], 50, 100)),
            pitch=int(np.clip(pitch, 48, 84)),
            start=t,
            end=t + note_length,
        )
        inst.notes.append(note)
        t += note_spacing

    response.instruments.append(inst)

    # --- Update memory ---
    MEMORY["last_motif"] = motif
    update_memory(MEMORY, notes, tempo, time_sig)

    return response


## Write Output

The output file can be imported into a DAW or opened in a MIDI player for evaluation.


In [7]:
response_midi = build_response(
    notes,
    action,
    source_instrument=instrument,
    features=features,
    start_time=midi.get_end_time()
)


In [8]:
combined = pretty_midi.PrettyMIDI(initial_tempo=TEMPO)
combined.instruments = midi.instruments + response_midi.instruments
OUTPUT_MIDI.parent.mkdir(parents=True, exist_ok=True)
combined.write(str(OUTPUT_MIDI))

print(f'Wrote {OUTPUT_MIDI}')
print('Detected features:', features)
print('Chosen action:', action)
print('Response instrument:', pretty_midi.program_to_instrument_name(response_midi.instruments[0].program))
print('Response note count:', len(response_midi.instruments[0].notes))


Wrote /Users/adrientalbot/Desktop/ai-jam-partner/data/output_midi/mvp_minimalist_response.mid
Detected features: {'note_count': 176, 'avg_pitch': 63.97727272727273, 'density': 'high', 'register': 'mid', 'bars': 32, 'motif': [72, 76, 79, 84]}
Chosen action: {'mode': 'contrast', 'response_density': 'low', 'bars': 32, 'latency_bars': 1, 'target_notes': 32}
Response instrument: Acoustic Grand Piano
Response note count: 32
